# Imports

In [1]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import TrainingArguments
import evaluate
from transformers import Trainer
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Current device:", torch.cuda.current_device())
else:
    print("Running on CPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4080
Current device: 0


In [3]:
# Load the XML data and convert it to a DataFrame
def xml_to_rows(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    rows = []

    for sentence in root.findall("sentence"):
        sid = sentence.get("id")
        text = sentence.find("text").text

        aspect_terms = sentence.find("aspectTerms")

        if aspect_terms is not None:
            for aspect in aspect_terms.findall("aspectTerm"):
                rows.append({
                    "id": sid,
                    "sentence": text,
                    "aspect": aspect.get("term"),
                    "polarity": aspect.get("polarity")
                })

    return pd.DataFrame(rows)

In [4]:
myXML = xml_to_rows("./Restaurants_Train_v2.xml")

In [5]:
myXML.shape

(3693, 4)

In [6]:
myXML.head()

,id,sentence,aspect,polarity
0,3121,But the staff was so horrible to us.,staff,negative
1,2777,"To be completely fair, the only redeeming fact...",food,positive
2,1634,"The food is uniformly exceptional, with a very...",food,positive
3,1634,"The food is uniformly exceptional, with a very...",kitchen,positive
4,1634,"The food is uniformly exceptional, with a very...",menu,neutral


In [7]:
csv_df = pd.read_csv("./Laptop_Train_v2.csv", encoding="utf-8")

csv_df = csv_df[[
    "id",
    "Sentence",
    "Aspect Term",
    "polarity"
]]

csv_df.columns = ["id", "sentence", "aspect", "polarity"]

In [8]:
csv_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [9]:
csv_df.shape

(2358, 4)

# Merge the datasets

In [10]:
final_df = pd.concat([csv_df, myXML], ignore_index=True)

In [11]:
final_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [12]:
final_df["sentence"].iloc[1]

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [13]:
final_df["sentence"].iloc[1]

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [14]:
final_df.shape

(6051, 4)

In [15]:
2358+3693

6051

# Preprocess

In [16]:
final_df["polarity"] = final_df["polarity"].str.strip()
final_df["aspect"] = final_df["aspect"].str.replace('"', '')

# Take 1000 of dataset

In [17]:
final_df = final_df.head(1000)

In [18]:
final_df.shape

(1000, 4)

# Model input

In [19]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

final_df["label"] = final_df["polarity"].map(label_map)

final_df["text"] = final_df["sentence"] + " [SEP] " + final_df["aspect"]

In [20]:
final_df = final_df.dropna(subset=["label"])
final_df["label"] = final_df["label"].astype(int)

# Train/test split

In [21]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42)

# Hugging Face dataset conversion

In [22]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [23]:
# train_dataset = train_dataset.remove_columns(["__index_level_0__"])
# test_dataset = test_dataset.remove_columns(["__index_level_0__"])


# train_dataset = train_dataset.cast_column("label", "int64")
# test_dataset = test_dataset.cast_column("label", "int64")

# Tokenization

In [24]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [25]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

# Format for PyTorch

In [26]:
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training setup

In [28]:
args = TrainingArguments(
    output_dir="./absa_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch"
)

# Trainer

In [29]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [30]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_macro = f1.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1_macro["f1"]
    }

In [31]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.698213,0.739796,0.539229
2,No log,0.657795,0.750000,0.678884
3,No log,0.600062,0.795918,0.744214
4,No log,0.603894,0.785714,0.740406


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=196, training_loss=0.6148311848543129, metrics={'train_runtime': 18.8306, 'train_samples_per_second': 166.538, 'train_steps_per_second': 10.409, 'total_flos': 206280919498752.0, 'train_loss': 0.6148311848543129, 'epoch': 4.0})

# Evaluations

In [33]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
No log,0.603894,4,0.785714,0.740406


{'eval_loss': 0.6038938164710999,
 'eval_accuracy': 0.7857142857142857,
 'eval_f1_macro': 0.7404060777195106}

In [34]:
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        70
           1       0.59      0.54      0.57        35
           2       0.84      0.87      0.85        91

    accuracy                           0.79       196
   macro avg       0.74      0.74      0.74       196
weighted avg       0.78      0.79      0.78       196



# inference

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()


text = "The service center is very slow"
aspect = "service center"

inputs = tokenizer(
    text + " [SEP] " + aspect,
    return_tensors="pt"
)

# 🔥 MOVE INPUTS TO SAME DEVICE AS MODEL
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

print(pred)

0
